# 02 · Silver Layer

The silver layer receives data from the bronze layer and applies **cleansing, normalisation and enrichment**,
producing tables ready for dimensional modelling in the gold layer.

**What this notebook does:**
1. Reads the current state of each bronze table
2. Applies quality transformations (nulls, types, string normalisation)
3. Resolves foreign keys (e.g., `color_id` to `color_name`)
4. Handles NULL values with defaults (e.g., `buying_group_name` to `'Unknown'`)
5. Computes derived measures (e.g., `backordered_quantity`, `line_total_excl_tax`)
6. Validates quality and referential integrity

---

**Tables produced:**

| Silver table | Bronze source | Key transformations |
|---|---|---|
| `stg_geography` | cities + state_provinces + countries | JOIN, NULL to 'Unknown' |
| `stg_customers` | customers + customer_categories + buying_groups + people + cities | JOINs, NULL to 'Unknown', trim |
| `stg_products` | stock_items + suppliers + colors + package_types + stock_groups | JOINs, NULL to 'N/A', many-to-many |
| `stg_employees` | people + cities | JOIN, filter is_employee |
| `stg_delivery_methods` | delivery_methods | trim |
| `stg_invoices` | invoices | direct copy |
| `stg_invoice_lines` | invoice_lines | line_total_excl_tax computed |
| `stg_orders` | orders | direct copy |
| `stg_order_lines` | order_lines | backordered_quantity computed |

## 1. Imports and connections

Database drivers, environment variables, and SQLAlchemy engines are configured to connect to the source and the data warehouse for silver processing.

In [ ]:
import pandas as pd
from datetime import datetime, timezone
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()

DWH_HOST     = os.getenv("DWH_HOST")
DWH_PORT     = int(os.getenv("DWH_PORT", 5432))
DWH_DB       = os.getenv("DWH_DB")
DWH_USER     = os.getenv("DWH_USER")
DWH_PASSWORD = os.getenv("DWH_PASSWORD")

engine_dwh = create_engine(
    f"postgresql+psycopg2://{DWH_USER}:{DWH_PASSWORD}@{DWH_HOST}:{DWH_PORT}/{DWH_DB}"
)

run_at = datetime.now(timezone.utc).replace(tzinfo=None)
print(f" Engine created.")
print(f"  DWH: {DWH_USER}@{DWH_HOST}:{DWH_PORT}/{DWH_DB}")
print(f"  Start: {run_at.strftime('%Y-%m-%d %H:%M:%S')} UTC")

## 2. Silver table creation

Silver tables are created in the DM to receive cleaned and transformed records from the bronze layer. They have the `stg_` prefix and are always truncated and reloaded (SCD Type 1).

In [ ]:
def table_exists(conn, schema, name):
    """Checks if a table already exists in the given schema."""
    return conn.execute(text(f"""
        SELECT 1 FROM information_schema.tables
        WHERE table_schema = '{schema}' AND table_name = '{name}'
    """)).fetchone() is not None

DDL_SILVER = {
"stg_geography": """
    CREATE TABLE IF NOT EXISTS silver.stg_geography (
        city_id               INT           NOT NULL,
        city_name             VARCHAR(100),
        state_province_name   VARCHAR(100),
        country_name          VARCHAR(100),
        sales_territory       VARCHAR(50),
        _loaded_at            TIMESTAMP     NOT NULL,
        _source_snapshot      INT           NOT NULL
    );
""",

"stg_customers": """
    CREATE TABLE IF NOT EXISTS silver.stg_customers (
        customer_id             INT           NOT NULL,
        customer_name           VARCHAR(100),
        customer_category_name  VARCHAR(50),
        buying_group_name       VARCHAR(50),
        bill_to_customer_name   VARCHAR(100),
        primary_contact_name    VARCHAR(100),
        phone_number            VARCHAR(50),
        website_url             VARCHAR(250),
        delivery_city_name      VARCHAR(100),
        city_id                 INT,
        is_on_credit_hold       BOOLEAN,
        _loaded_at              TIMESTAMP     NOT NULL,
        _source_snapshot        INT           NOT NULL
    );
""",

"stg_products": """
    CREATE TABLE IF NOT EXISTS silver.stg_products (
        stock_item_id       INT             NOT NULL,
        stock_item_name     VARCHAR(100),
        color_name          VARCHAR(50),
        unit_package_name   VARCHAR(50),
        outer_package_name  VARCHAR(50),
        brand               VARCHAR(50),
        size                VARCHAR(50),
        lead_time_days      INT,
        quantity_per_outer  INT,
        is_chiller_stock    BOOLEAN,
        tax_rate            NUMERIC(5,2),
        unit_price          NUMERIC(10,2),
        stock_group_name    VARCHAR(100),
        _loaded_at          TIMESTAMP       NOT NULL,
        _source_snapshot    INT             NOT NULL
    );
""",

"stg_employees": """
    CREATE TABLE IF NOT EXISTS silver.stg_employees (
        person_id        INT           NOT NULL,
        full_name        VARCHAR(100),
        preferred_name   VARCHAR(50),
        is_salesperson   BOOLEAN,
        city_id          INT,
        _loaded_at       TIMESTAMP     NOT NULL,
        _source_snapshot INT           NOT NULL
    );
""",

"stg_delivery_methods": """
    CREATE TABLE IF NOT EXISTS silver.stg_delivery_methods (
        delivery_method_id   INT          NOT NULL,
        delivery_method_name VARCHAR(50),
        _loaded_at           TIMESTAMP    NOT NULL,
        _source_snapshot     INT          NOT NULL
    );
""",

"stg_invoices": """
    CREATE TABLE IF NOT EXISTS silver.stg_invoices (
        invoice_id            INT   NOT NULL,
        customer_id           INT,
        bill_to_customer_id   INT,
        order_id              INT,
        delivery_method_id    INT,
        salesperson_person_id INT,
        invoice_date          DATE,
        _loaded_at            TIMESTAMP NOT NULL
    );
""",

"stg_invoice_lines": """
    CREATE TABLE IF NOT EXISTS silver.stg_invoice_lines (
        invoice_line_id     INT            NOT NULL,
        invoice_id          INT,
        stock_item_id       INT,
        quantity            INT,
        unit_price          NUMERIC(18,2),
        tax_amount          NUMERIC(18,2),
        line_total_excl_tax NUMERIC(18,2),
        line_profit         NUMERIC(18,2),
        _loaded_at          TIMESTAMP      NOT NULL
    );
""",

"stg_orders": """
    CREATE TABLE IF NOT EXISTS silver.stg_orders (
        order_id              INT   NOT NULL,
        customer_id           INT,
        salesperson_person_id INT,
        order_date            DATE,
        expected_delivery_date DATE,
        delivery_method_id    INT,
        _loaded_at            TIMESTAMP NOT NULL
    );
""",

"stg_order_lines": """
    CREATE TABLE IF NOT EXISTS silver.stg_order_lines (
        order_line_id        INT            NOT NULL,
        order_id             INT,
        stock_item_id        INT,
        ordered_quantity     INT,
        picked_quantity      INT,
        backordered_quantity INT,
        unit_price           NUMERIC(18,2),
        tax_rate             NUMERIC(5,2),
        _loaded_at           TIMESTAMP      NOT NULL
    );
""",
}

TRUNCATE_SILVER = """
    TRUNCATE TABLE
        silver.stg_geography,
        silver.stg_customers,
        silver.stg_products,
        silver.stg_employees,
        silver.stg_delivery_methods,
        silver.stg_invoices,
        silver.stg_invoice_lines,
        silver.stg_orders,
        silver.stg_order_lines;
"""

with engine_dwh.begin() as conn:
    created, skipped = [], []
    for name, ddl in DDL_SILVER.items():
        existed = table_exists(conn, "silver", name)
        conn.execute(text(ddl))
        (skipped if existed else created).append(name)
    if skipped:
        conn.execute(text(TRUNCATE_SILVER))

if created:
    print("Tables created:")
    for name in created:
        print(f"   silver.{name}")
if skipped:
    print("Tables already exist — truncated for reload:")
    for name in skipped:
        print(f"    silver.{name}")

## 3. Helper functions

Helper functions are defined to support data transformations, lookups, and table operations used by the silver pipeline.

In [ ]:
def read_active_snapshot(table_name: str) -> pd.DataFrame:
    """Reads the active state of the latest snapshot from a bronze dimension table."""
    sql = f"""
        SELECT *
        FROM   bronze.{table_name}
        WHERE  _snapshot_id = (SELECT MAX(_snapshot_id) FROM bronze.{table_name})
          AND  _change_op  != 'DELETE'
    """
    with engine_dwh.connect() as conn:
        return pd.read_sql(text(sql), conn)


def clean_str(s: pd.Series) -> pd.Series:
    """Strips whitespace; empty strings become None (NULL in SQL)."""
    return s.str.strip().replace("", None)


print("Helper functions defined.")

## 4. `stg_geography`

The geography staging table is built from bronze geography data and normalized for reuse by downstream silver transformations.

Transformations:
- JOIN Cities + StateProvinces + Countries
- `sales_territory`: NULL to `'Unknown'`
- Trim on all text columns

In [ ]:
df_cities = read_active_snapshot("cities")
df_sp     = read_active_snapshot("stateprovinces")
df_ctr    = read_active_snapshot("countries")

snap_id = int(df_cities["_snapshot_id"].max())

df = (
    df_cities[["cityid", "cityname", "stateprovinceid"]]
    .merge(
        df_sp[["stateprovinceid", "stateprovincename", "countryid", "salesterritory"]],
        on="stateprovinceid", how="left"
    )
    .merge(
        df_ctr[["countryid", "ountryname"]],
        on="countryid", how="left"
    )
)

df = df.rename(columns={
    "cityid": "city_id",
    "cityname": "city_name",
    "stateprovincename": "state_province_name",
    "ountryname": "country_name",
    "salesterritory": "sales_territory",
})

for col in ["city_name", "state_province_name", "country_name"]:
    df[col] = clean_str(df[col])

df["sales_territory"] = df["sales_territory"].fillna("Unknown").str.strip()

silver_geography = df[["city_id", "city_name", "state_province_name",
                        "country_name", "sales_territory"]].copy()
silver_geography["_loaded_at"]       = run_at
silver_geography["_source_snapshot"] = snap_id

with engine_dwh.begin() as conn:
    silver_geography.to_sql("stg_geography", conn, schema="silver",
                            if_exists="append", index=False)

print(f" stg_geography   — {len(silver_geography)} records  (snapshot {snap_id})")
silver_geography.head(3)

## 5. `stg_customers`

The customer staging table cleans and enriches customer data to prepare it for silver-level analytics and dimension building.

Transformations:
- `color_name` via LEFT JOIN (NULL to `'N/A'`)
- `unit_package_name` and `outer_package_name` via JOIN with `package_types`
- `stock_group_name` via many-to-many JOIN with `stock_item_stock_groups` + `stock_groups`

In [ ]:
df_cust   = read_active_snapshot("customers")
df_cat    = read_active_snapshot("customercategories")
df_bg     = read_active_snapshot("buyinggroups")
df_people = read_active_snapshot("people")
df_cities = read_active_snapshot("cities")

snap_id = int(df_cust["_snapshot_id"].max())

df = (
    df_cust[[
        "customerid", "customername", "billtocustomerid",
        "customercategoryid", "buyinggroupid",
        "primarycontactpersonid", "deliverycityid",
        "isoncredithold", "phonenumber", "websiteurl"
    ]]
    # category
    .merge(df_cat[["customercategoryid", "customercategoryname"]],
           on="customercategoryid", how="left")
    # buying group
    .merge(df_bg[["buyinggroupid", "buyinggroupname"]],
           on="buyinggroupid", how="left")
    # primary contact
    .merge(df_people[["personid", "fullname"]].rename(
               columns={"personid": "primarycontactpersonid",
                        "fullname": "primary_contact_name"}),
           on="primarycontactpersonid", how="left")
    # delivery city
    .merge(df_cities[["cityid", "cityname"]].rename(
               columns={"cityid": "deliverycityid",
                        "cityname": "delivery_city_name"}),
           on="deliverycityid", how="left")
)

# Bill-to customer name (self-join)
bill_to = df_cust[["customerid", "customername"]].rename(
    columns={"customerid": "billtocustomerid", "customername": "bill_to_customer_name"}
)
df = df.merge(bill_to, on="billtocustomerid", how="left")

# Nulls and cleansing
df["buyinggroupname"] = df["buyinggroupname"].fillna("Unknown")
for col in ["customername", "primary_contact_name", "delivery_city_name",
            "bill_to_customer_name", "phonenumber", "websiteurl"]:
    df[col] = clean_str(df[col].astype(str).replace("nan", None))

# isoncredithold: SMALLINT to bool
df["isoncredithold"] = df["isoncredithold"].apply(lambda x: bool(x) if pd.notna(x) else None)

silver_customers = df[[
    "customerid", "customername", "customercategoryname",
    "buyinggroupname", "bill_to_customer_name", "primary_contact_name",
    "phonenumber", "websiteurl", "delivery_city_name",
    "deliverycityid", "isoncredithold"
]].rename(columns={
    "customerid": "customer_id",
    "customername": "customer_name",
    "customercategoryname": "customer_category_name",
    "buyinggroupname": "buying_group_name",
    "phonenumber": "phone_number",
    "websiteurl": "website_url",
    "deliverycityid": "city_id",
    "isoncredithold": "is_on_credit_hold",
}).copy()

silver_customers["_loaded_at"]       = run_at
silver_customers["_source_snapshot"] = snap_id

with engine_dwh.begin() as conn:
    silver_customers.to_sql("stg_customers", conn, schema="silver",
                            if_exists="append", index=False)

print(f" stg_customers   — {len(silver_customers)} records  (snapshot {snap_id})")
silver_customers.head(3)

## 6. `stg_products`

The product staging table standardizes stock item attributes and reference data for silver consumption.

Transformations:
- `supplier_name` via JOIN with `suppliers`
- `color_name` via LEFT JOIN (NULL to `'N/A'`)
- `unit_package_name` and `outer_package_name` via JOIN with `package_types`
- `stock_group_name` via many-to-many JOIN with `stock_item_stock_groups` + `stock_groups`

In [ ]:
df_si    = read_active_snapshot("stockitems")
df_col   = read_active_snapshot("colors")
df_pkg   = read_active_snapshot("packagetypes")

snap_id = int(df_si["_snapshot_id"].max())

# Unit package
unit_pkg = df_pkg[["packagetypeid", "packagetypename"]].rename(
    columns={"packagetypeid": "unitpackageid", "packagetypename": "unit_package_name"}
)
# Outer package
outer_pkg = df_pkg[["packagetypeid", "packagetypename"]].rename(
    columns={"packagetypeid": "outerpackageid", "packagetypename": "outer_package_name"}
)

df = (
    df_si[[
        "stockitemid", "stockitemname", "colorid",
        "unitpackageid", "outerpackageid", "brand", "size",
        "leadtimedays", "quantityperouter", "ischillerstock",
        "taxrate", "unitprice"
    ]]
    .merge(df_col[["colorid", "colorname"]], on="colorid", how="left")
    .merge(unit_pkg, on="unitpackageid", how="left")
    .merge(outer_pkg, on="outerpackageid", how="left")
)

# stockitemstockgroups table does not exist in the source
df["stock_group_name"] = "Unknown"

df["colorname"]    = df["colorname"].fillna("N/A")
df["brand"]        = df["brand"].fillna("N/A")
df["size"]         = df["size"].fillna("N/A")

# ischillerstock: SMALLINT to bool
df["ischillerstock"] = df["ischillerstock"].apply(lambda x: bool(x) if pd.notna(x) else None)

for col in ["stockitemname", "unit_package_name", "outer_package_name"]:
    df[col] = clean_str(df[col].astype(str).replace("nan", None))

silver_products = df[[
    "stockitemid", "stockitemname", "colorname",
    "unit_package_name", "outer_package_name", "brand", "size",
    "leadtimedays", "quantityperouter", "ischillerstock",
    "taxrate", "unitprice", "stock_group_name"
]].rename(columns={
    "stockitemid": "stock_item_id",
    "stockitemname": "stock_item_name",
    "colorname": "color_name",
    "leadtimedays": "lead_time_days",
    "quantityperouter": "quantity_per_outer",
    "ischillerstock": "is_chiller_stock",
    "taxrate": "tax_rate",
    "unitprice": "unit_price",
}).copy()

silver_products["_loaded_at"]       = run_at
silver_products["_source_snapshot"] = snap_id

with engine_dwh.begin() as conn:
    silver_products.to_sql("stg_products", conn, schema="silver",
                           if_exists="append", index=False)

print(f" stg_products    — {len(silver_products)} records  (snapshot {snap_id})")
silver_products.head(3)

## 7. `stg_employees`

The employee staging table transforms workforce and salesperson information for the silver layer.

Transformations:
- Loads all records from the `people` table (includes employees and salespersons)
- `city_id` for later lookup in `dim_geography` (when available)

In [ ]:
df_ppl  = read_active_snapshot("people")
snap_id = int(df_ppl["_snapshot_id"].max())

for col in ["fullname", "preferredname"]:
    df_ppl[col] = clean_str(df_ppl[col])

# issalesperson: SMALLINT to bool
df_ppl["issalesperson"] = df_ppl["issalesperson"].apply(lambda x: bool(x) if pd.notna(x) else None)

silver_employees = df_ppl[[
    "personid", "fullname", "preferredname", "issalesperson"
]].rename(columns={
    "personid": "person_id",
    "fullname": "full_name",
    "preferredname": "preferred_name",
    "issalesperson": "is_salesperson",
}).copy()
silver_employees["city_id"] = None

silver_employees["_loaded_at"]       = run_at
silver_employees["_source_snapshot"] = snap_id

with engine_dwh.begin() as conn:
    silver_employees.to_sql("stg_employees", conn, schema="silver",
                            if_exists="append", index=False)

print(f" stg_employees   — {len(silver_employees)} records  (snapshot {snap_id})")
silver_employees.head(3)

## 8. `stg_delivery_methods`

The delivery methods staging table normalizes delivery channel metadata for use in silver transformations.

Transformation: 
- Name trim.

In [ ]:
df_dm   = read_active_snapshot("deliverymethods")
snap_id = int(df_dm["_snapshot_id"].max())

df_dm["deliverymethodname"] = clean_str(df_dm["deliverymethodname"])

silver_dm = df_dm[["deliverymethodid", "deliverymethodname"]].rename(columns={
    "deliverymethodid": "delivery_method_id",
    "deliverymethodname": "delivery_method_name",
}).copy()
silver_dm["_loaded_at"]       = run_at
silver_dm["_source_snapshot"] = snap_id

with engine_dwh.begin() as conn:
    silver_dm.to_sql("stg_delivery_methods", conn, schema="silver",
                     if_exists="append", index=False)

print(f" stg_delivery_methods — {len(silver_dm)} records  (snapshot {snap_id})")
silver_dm

## 9. `stg_invoices`

The invoice staging table loads transactional invoice headers with their business keys and date metadata.

Direct copy from bronze, no additional transformations.

In [ ]:
with engine_dwh.connect() as conn:
    df_inv = pd.read_sql(text("SELECT * FROM bronze.invoices"), conn)

silver_inv = df_inv[[
    "invoiceid", "customerid", "billtocustomerid", "orderid",
    "deliverymethodid", "salespersonpersonid", "invoicedate"
]].rename(columns={
    "invoiceid": "invoice_id",
    "customerid": "customer_id",
    "billtocustomerid": "bill_to_customer_id",
    "orderid": "order_id",
    "deliverymethodid": "delivery_method_id",
    "salespersonpersonid": "salesperson_person_id",
    "invoicedate": "invoice_date",
}).copy()
silver_inv["_loaded_at"] = run_at

with engine_dwh.begin() as conn:
    silver_inv.to_sql("stg_invoices", conn, schema="silver",
                      if_exists="append", index=False)

print(f" stg_invoices    — {len(silver_inv)} records")
silver_inv.head(3)

## 10. `stg_invoice_lines`

The invoice lines staging table loads invoice detail rows including pricing, quantity, and tax information.

Transformations:
- `line_total_excl_tax` = `quantity × unit_price` (derived measure)

In [ ]:
with engine_dwh.connect() as conn:
    df_il = pd.read_sql(text("SELECT * FROM bronze.invoicelines"), conn)

df_il["line_total_excl_tax"] = (
    pd.to_numeric(df_il["quantity"],    errors="coerce") *
    pd.to_numeric(df_il["unitprice"],   errors="coerce")
).round(2)

silver_il = df_il[[
    "invoicelineid", "invoiceid", "stockitemid",
    "quantity", "unitprice", "taxamount",
    "line_total_excl_tax", "lineprofit"
]].rename(columns={
    "invoicelineid": "invoice_line_id",
    "invoiceid": "invoice_id",
    "stockitemid": "stock_item_id",
    "unitprice": "unit_price",
    "taxamount": "tax_amount",
    "lineprofit": "line_profit",
}).copy()
silver_il["_loaded_at"] = run_at

with engine_dwh.begin() as conn:
    silver_il.to_sql("stg_invoice_lines", conn, schema="silver",
                     if_exists="append", index=False)

print(f" stg_invoice_lines — {len(silver_il)} records")
silver_il.head(3)

## 11. `stg_orders`

The order staging table loads order headers and related metadata for the silver layer.

Direct copy from bronze, no additional transformations.

In [ ]:
with engine_dwh.connect() as conn:
    df_ord = pd.read_sql(text("SELECT * FROM bronze.orders"), conn)
    # orders doesn't have deliverymethodid — get it via invoices
    df_inv_dm = pd.read_sql(text("""
        SELECT DISTINCT orderid, deliverymethodid
        FROM bronze.invoices
        WHERE orderid IS NOT NULL
    """), conn)

df_ord = df_ord.merge(df_inv_dm, on="orderid", how="left")

silver_ord = df_ord[[
    "orderid", "customerid", "salespersonpersonid",
    "orderdate", "expecteddeliverydate", "deliverymethodid"
]].rename(columns={
    "orderid": "order_id",
    "customerid": "customer_id",
    "salespersonpersonid": "salesperson_person_id",
    "orderdate": "order_date",
    "expecteddeliverydate": "expected_delivery_date",
    "deliverymethodid": "delivery_method_id",
}).copy()
silver_ord["_loaded_at"] = run_at

with engine_dwh.begin() as conn:
    silver_ord.to_sql("stg_orders", conn, schema="silver",
                      if_exists="append", index=False)

print(f" stg_orders      — {len(silver_ord)} records")
silver_ord.head(3)

## 12. `stg_order_lines`

The order lines staging table loads order detail rows for downstream order reporting and analytics.

Transformations:
- `ordered_quantity` ← `quantity` (renamed for clarity)
- `backordered_quantity` = `quantity − picked_quantity` (derived measure)

In [ ]:
with engine_dwh.connect() as conn:
    df_ol = pd.read_sql(text("SELECT * FROM bronze.orderlines"), conn)

df_ol = df_ol.rename(columns={"quantity": "ordered_quantity"})

df_ol["pickedquantity"]       = pd.to_numeric(df_ol["pickedquantity"],  errors="coerce").fillna(0).astype(int)
df_ol["ordered_quantity"]     = pd.to_numeric(df_ol["ordered_quantity"], errors="coerce").fillna(0).astype(int)
df_ol["backordered_quantity"] = (df_ol["ordered_quantity"] - df_ol["pickedquantity"]).clip(lower=0)

# Validation: backordered_quantity cannot be negative
anomalies = df_ol[df_ol["backordered_quantity"] < 0]
if not anomalies.empty:
    print(f"⚠ {len(anomalies)} rows with negative backordered_quantity. Corrected to 0.")

silver_ol = df_ol[[
    "orderlineid", "orderid", "stockitemid",
    "ordered_quantity", "pickedquantity", "backordered_quantity",
    "unitprice", "taxrate"
]].rename(columns={
    "orderlineid": "order_line_id",
    "orderid": "order_id",
    "stockitemid": "stock_item_id",
    "pickedquantity": "picked_quantity",
    "unitprice": "unit_price",
    "taxrate": "tax_rate",
}).copy()
silver_ol["_loaded_at"] = run_at

with engine_dwh.begin() as conn:
    silver_ol.to_sql("stg_order_lines", conn, schema="silver",
                     if_exists="append", index=False)

print(f" stg_order_lines  — {len(silver_ol)} records")
silver_ol.head(3)

## 13. Quality checks

Quality checks are run to validate table counts and ensure the silver transformation logic produced the expected results.

In [ ]:
silver_tables = [
    "stg_geography", "stg_customers", "stg_products", "stg_employees",
    "stg_delivery_methods", "stg_invoices", "stg_invoice_lines",
    "stg_orders", "stg_order_lines",
]

rows = []
with engine_dwh.connect() as conn:
    for t in silver_tables:
        cnt = conn.execute(text(f"SELECT COUNT(*) FROM silver.{t}")).scalar()
        rows.append({"table": f"silver.{t}", "records": cnt})

df_summary = pd.DataFrame(rows)
print("Silver row counts:")
print(df_summary.to_string(index=False))

# Check NULLs in critical columns
print("\nNULL check on critical columns:")
checks = [
    ("stg_customers",     "customer_id"),
    ("stg_products",      "stock_item_id"),
    ("stg_employees",     "person_id"),
    ("stg_invoice_lines", "invoice_id"),
    ("stg_order_lines",   "order_id"),
]
with engine_dwh.connect() as conn:
    for tbl, col in checks:
        nulls = conn.execute(text(f"SELECT COUNT(*) FROM silver.{tbl} WHERE {col} IS NULL")).scalar()
        status = "" if nulls == 0 else f"⚠ {nulls} NULLs"
        print(f"  {status}  silver.{tbl}.{col}")

print("\n Silver completed. Next step: run 03_gold.ipynb")